# Using the metrics suite

You have designs from one or more models and want to know what they are worth. The
whole workflow is one registry and one object:

| You want to... | You call |
|---|---|
| see every metric and what it measures | `METRICS.explain()` |
| score your models | `Board(problem, reference=..., train=...).evaluate(models)` |
| read the result | `board.explain()`, `board.rank("mmd")` |
| change how per-design values are combined | `board.evaluate(models, aggregation="median")` |
| ask the same questions in another space | `board.evaluate(models, space="pca")` |
| score published checkpoints by name | `Board.load("beams2d", ["cgan_cnn_2d", "vqgan"])` |
| add a metric of your own | `@register_metric(...)` on a function |

Everything below runs in seconds on a toy problem. With a real EngiBench problem the
calls are identical.

## 1. What metrics exist, and what does each one measure?

In [1]:
import warnings

import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="gymnasium")
pd.set_option("display.max_colwidth", None)

from engiopt.evaluation.registry import METRICS
import engiopt.evaluation.metrics  # noqa: F401  -- importing registers the built-in metrics

METRICS.explain()

,question,family,direction,cost
mmd,"Taken as a set, how different are the generated designs from the reference optimal designs?",distribution,lower is better,cheap
dpp,"How spread out are the generated designs, on a 0-to-1 scale where 1 is a perfectly diverse set?",diversity,higher is better,cheap
train_distance,How far is each generated design from the nearest design the model was trained on?,memorization,diagnostic,cheap
copy_rate,What fraction of the generated designs are copies of a design the model could have seen?,memorization,diagnostic,cheap
cond_sens,"When the same model is given different conditions, does its output change?",conditions,diagnostic,cheap
viol,What fraction of the generated designs violate the problem's constraints?,feasibility,lower is better,cheap
iog,"How much worse than the reference optimum is the generated design, exactly as generated?",performance,lower is better,expensive
cog,"Starting the optimizer from the generated design, how much worse than optimal is it, summed over every optimizer step?",performance,lower is better,expensive
fog,"Starting the optimizer from the generated design, how much worse than optimal is it once the optimizer finishes?",performance,lower is better,expensive
per_condition_distance,"For each set of conditions, how far is the generated design from the reference optimum for those same conditions?",conditions,lower is better,cheap


Each row is a question written so it makes sense without knowing the metric's name,
plus which way is better. Some say **diagnostic**: those columns tell you whether the
*other* columns can be trusted — a lookup table posts a perfect `mmd`, and only
`copy_rate` says so — and they are never ranked on.

`cog` is worth a second look because it is the least obvious: the optimizer is started
from the generated design, and `objective(step) − objective(reference optimum)` is
summed over every step it takes. It is the area under the optimality-gap curve, so it
mixes how far from optimal the start was (`iog`) with how many steps were needed.

## 2. A toy problem

Every EngiBench dataset has the same structure: **one optimal design per set of
conditions**, split into training and withheld test conditions. The toy keeps exactly
that and nothing else — the optimal design for condition `c` is a field with mean `c`.
With a real problem, `problem` is an EngiBench `Problem` and the two arrays come from
its dataset splits.

In [2]:
import numpy as np
from gymnasium import spaces


class ToyProblem:
    """All the cheap metrics need from a problem: a design space and a constraint check."""

    design_space = spaces.Box(0.0, 1.0, shape=(8, 10), dtype=np.float64)

    def check_constraints(self, design, config):
        class Violations:
            violations = [] if self.design_space.contains(design) else ["outside design space"]

        return Violations()


rng = np.random.default_rng(0)
problem = ToyProblem()


def optimal_design(c):
    return np.clip(np.full((8, 10), c) + rng.normal(0, 0.01, (8, 10)), 0, 1)


train_conditions = rng.uniform(0.2, 0.8, 64)
test_conditions = rng.uniform(0.2, 0.8, 16)
TRAIN = np.stack([optimal_design(c) for c in train_conditions])       # what models are fitted on
REFERENCE = np.stack([optimal_design(c) for c in test_conditions])    # withheld; what models are scored against

## 3. Five models

Three you might really train; two are **constructions** whose right score you know in
advance, so they test the metrics rather than the models.

| Model | What it does |
|---|---|
| `honest` | the optimal design for each test condition |
| `lookup_table` | nearest neighbour over the training set; never sees a withheld design |
| `unconditional` | real training designs, ignoring the conditions |
| `permuted_conditions` | the correct withheld designs shuffled across conditions — every design a true optimum, every one for the wrong conditions |
| `noise` | uniform noise |

In [3]:
def lookup_table(c):
    return TRAIN[np.argmin(np.abs(train_conditions - c))]


MODELS = {
    "honest": np.stack([optimal_design(c) for c in test_conditions]),
    "lookup_table": np.stack([lookup_table(c) for c in test_conditions]),
    "unconditional": TRAIN[rng.integers(0, len(TRAIN), len(REFERENCE))],
    "permuted_conditions": REFERENCE[rng.permutation(len(REFERENCE))],
    "noise": rng.random(REFERENCE.shape),
}

## 4. Score them

In [4]:
from engiopt.evaluation.board import Board

board = Board(problem, reference=REFERENCE, train=TRAIN)
board.evaluate(MODELS)

,mmd,dpp,train_distance,copy_rate,cond_sens,viol,per_condition_distance,volume_error,coverage,vendi,generation_seconds,n_parameters,train_minutes
honest,6.366160e-04,0.035297,0.014761,0.0,NaN,0.0,0.014059,NaN,1.0000,3.254471,NaN,NaN,NaN
lookup_table,1.037788e-03,0.004358,0.000000,1.0,NaN,0.0,0.014891,NaN,1.0000,3.241546,NaN,NaN,NaN
unconditional,1.012030e-01,0.006445,0.000000,1.0,NaN,0.0,0.180742,NaN,0.9375,2.605583,NaN,NaN,NaN
permuted_conditions,-1.110223e-16,0.033897,0.014753,1.0,NaN,0.0,0.176872,NaN,1.0000,3.248472,NaN,NaN,NaN
noise,4.239221e-01,0.998588,0.285783,0.0,NaN,0.0,0.332793,NaN,0.0000,15.976387,NaN,NaN,NaN
reference (split-half),2.798658e-01,0.090922,0.014603,0.0,NaN,0.0,NaN,NaN,0.5000,2.868064,NaN,NaN,NaN


One row per model, one column per metric — and a last row that is **not a model**.
`reference (split-half)` is half of the withheld optimal designs scored against the
other half: what real, correct designs score on every column, on this problem. An
`mmd` of 0.0006 means nothing on its own; next to what real designs score, it does.

Ask the board to read itself:

In [5]:
board.explain()

,question,direction,picks,real designs score
mmd,"Taken as a set, how different are the generated designs from the reference optimal designs?",lower is better,permuted_conditions,0.279866
dpp,"How spread out are the generated designs, on a 0-to-1 scale where 1 is a perfectly diverse set?",higher is better,noise,0.090922
train_distance,How far is each generated design from the nearest design the model was trained on?,diagnostic,,0.014603
copy_rate,What fraction of the generated designs are copies of a design the model could have seen?,diagnostic,,0.000000
cond_sens,"When the same model is given different conditions, does its output change?",diagnostic,,NaN
viol,What fraction of the generated designs violate the problem's constraints?,lower is better,"tie: honest, lookup_table, unconditional, permuted_conditions, noise",0.000000
per_condition_distance,"For each set of conditions, how far is the generated design from the reference optimum for those same conditions?",lower is better,honest,NaN
volume_error,How far is each design's material fraction from the one its conditions requested?,lower is better,,NaN
coverage,What fraction of the reference optima have a generated design nearby?,higher is better,"tie: honest, lookup_table, permuted_conditions",0.500000
vendi,"How many genuinely distinct designs are in the generated set, as an effective count?",higher is better,noise,2.868064


## 5. What the board is telling you

**`mmd` picks `permuted_conditions`.** It is handed the correct withheld optima with the
conditions scrambled, and the distribution metric rates it *first* — a perfect 0.0,
better than the honest model and better than real designs against real designs. MMD
compares two *sets*; it has no access to which design was produced for which
conditions. Every set-level metric shares this by construction.

**`lookup_table` scores almost as well as `honest` on `mmd`, having never seen a
withheld design.** Not by cheating: training and test optima come from the same
distribution, and a distribution metric only compares distributions. A lookup table is a
legitimately strong baseline; what is not legitimate is reading its `mmd` as evidence
it learned anything.

**`copy_rate` is 1.0 for all three constructions** and `novelty` is 0.0. One diagnostic
column separates every construction from the honest model. That is what the
`memorization` and `conditions` families are for, and why a board that reports only
distribution metrics cannot be trusted.

**`per_condition_distance` is the column that catches `permuted_conditions`.** It
compares design `i` with the reference optimum for conditions `i`, so the correct
designs in the wrong order score badly here and perfectly on `mmd`. Read the two
together.

**`dpp` and `vendi` both measure diversity; prefer `vendi`.** `dpp` is now the n-th
root of the kernel determinant, on a 0-to-1 scale (the raw determinant older boards
published reads 10⁻²⁰ on every real board). But a determinant still rises when a
collapsed set is jittered — it rewards noise as diversity — and `vendi`, an effective
count of distinct designs, does not.

**`cond_sens` is `NaN`.** It asks whether the output *changes* when the conditions
change, which needs the live model re-sampled; a saved array cannot answer it.
`Board` scores designs; `Evaluator.score(generator)` scores a model and fills it in.

## 6. Rank — but only on a column that can be ranked

In [6]:
board.rank("mmd")

,mmd,dpp,train_distance,copy_rate,cond_sens,viol,per_condition_distance,volume_error,coverage,vendi,generation_seconds,n_parameters,train_minutes
permuted_conditions,-1.110223e-16,0.033897,0.014753,1.0,NaN,0.0,0.176872,NaN,1.0000,3.248472,NaN,NaN,NaN
honest,6.366160e-04,0.035297,0.014761,0.0,NaN,0.0,0.014059,NaN,1.0000,3.254471,NaN,NaN,NaN
lookup_table,1.037788e-03,0.004358,0.000000,1.0,NaN,0.0,0.014891,NaN,1.0000,3.241546,NaN,NaN,NaN
unconditional,1.012030e-01,0.006445,0.000000,1.0,NaN,0.0,0.180742,NaN,0.9375,2.605583,NaN,NaN,NaN
reference (split-half),2.798658e-01,0.090922,0.014603,0.0,NaN,0.0,NaN,NaN,0.5000,2.868064,NaN,NaN,NaN
noise,4.239221e-01,0.998588,0.285783,0.0,NaN,0.0,0.332793,NaN,0.0000,15.976387,NaN,NaN,NaN


In [7]:
try:
    board.rank("copy_rate")
except ValueError as e:
    print(e)

'copy_rate' is a diagnostic: it says whether the other columns mean what they look like. Read it beside them; do not rank on it.


## 7. Changing how a column is aggregated

`iog`, `cog`, `fog`, `train_distance`, `per_condition_distance` and the other per-design
metrics each produce **one value per design**.
Which single number that becomes is a policy, not a measurement: a mean is dragged by
one diverged design, a median is not. It is an argument to `evaluate` — and, for a
published board, a field of the frozen spec recorded in every row — so there are no
`*_median` columns.

In [8]:
one_diverged = MODELS["honest"].copy()
one_diverged[0] = rng.random((8, 10))  # fifteen good designs and one that went wrong

compare = {"honest": MODELS["honest"], "one_diverged": one_diverged}
pd.DataFrame({
    "train_distance (mean)": board.evaluate(compare)["train_distance"],
    "train_distance (median)": board.evaluate(compare, aggregation="median")["train_distance"],
})

,train_distance (mean),train_distance (median)
honest,0.014761,0.014655
one_diverged,0.032199,0.015012
reference (split-half),0.014603,0.015140


## 8. Asking the same questions in another space

Where a metric is computed is also an argument, not a property of the metric. `pca`
projects every design onto the leading principal components of the reference designs
before scoring, and suffixes the columns. Metrics that need the actual designs — a
constraint check, a copy corpus — are skipped there, because a constraint check on PCA
coordinates would be meaningless.

This is the control that asks whether a *learned* space is needed at all: if PCA of the
data answers the same questions, any rotation of the same width would do. Learned
latent spaces (`space="lv"`) arrive with the LVAE.

In [9]:
board.evaluate(MODELS, space="pca", width=8)

,mmd@pca,dpp@pca,per_condition_distance@pca,coverage@pca,vendi@pca
honest,0.000124,0.007646,0.026240,1.0000,3.168944
lookup_table,0.000206,0.001569,0.029576,1.0000,3.182097
unconditional,0.101071,0.002117,0.569443,0.9375,2.543748
permuted_conditions,0.000000,0.022450,0.557238,1.0000,3.220545
noise,0.181628,0.262513,0.588434,0.2500,4.577680
reference (split-half),0.279983,0.071127,NaN,0.5000,2.849693


## 9. Registering a metric of your own

A metric is a function of an `EvaluationContext`, a family, a cost, and a direction.
The first line of its docstring is the question readers see. `ctx.reduce` applies
whatever aggregation the caller chose, so you never hard-code a mean.

In [10]:
from engiopt.evaluation.registry import register_metric


@register_metric("density_ratio", family="feasibility", cost="cheap", higher_is_better=None)
def density_ratio(ctx):
    """How much material do the generated designs use, relative to the reference optima?"""
    return float(ctx.reduce(ctx.gen_flat.mean(axis=1)) / ctx.ref_flat.mean())


board.evaluate(MODELS, metrics=["mmd", "density_ratio"])
board.explain()

,question,direction,picks,real designs score
mmd,"Taken as a set, how different are the generated designs from the reference optimal designs?",lower is better,permuted_conditions,0.279866
density_ratio,"How much material do the generated designs use, relative to the reference optima?",diagnostic,,0.766460


It appears on the next board with its question and direction attached — nothing else
to wire up. Declared diagnostic here, because more material is not better or worse.

Notice the reference row: two random halves of the *same* real designs do not score
1.0 against each other. On eight designs a side, that gap is the sampling noise of this
column on this problem — and it is the scale against which every model's deviation from
1.0 has to be read. That is what the reference row is for.

## 10. Not in this notebook

- **Physics.** `iog`, `cog`, `fog`, `calls_to_settle`, `gap_after_calls`,
  `reaches_reference_rate` and `first_call_gain` need a simulator. With a real problem,
  `Board.load("beams2d", ["cgan_cnn_2d", "vqgan"], expensive=True)` runs them.
- **Latent spaces.** `space="lv"` needs a verified autoencoder pinned by the spec.
- **Uncertainty.** Confidence intervals beside every value. On sixteen samples, most
  gaps on this board are smaller than their own error bars.